## Importing the libraries

In [2]:
import pandas as pd

## Load datasets

In [3]:
math_df = pd.read_csv('student-mat.csv', sep=';')
por_df = pd.read_csv('student-por.csv', sep=';')
print("Math shape:", math_df.shape)
print("Portugese shpae:", por_df.shape)

Math shape: (395, 33)
Portugese shpae: (649, 33)


## Datasets cleaning

In [4]:
### Adding Subject column
math_df['subject'] = 'math'
por_df['subject'] = 'portugese'

In [5]:
### Merging the two dataframes
combined_df = pd.concat([math_df, por_df], axis=0)

### Reseting the index
combined_df.reset_index(drop=True, inplace=True)

print("Combined shape:", combined_df.shape)
combined_df.head()

Combined shape: (1044, 34)


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3,subject
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,3,4,1,1,3,6,5,6,6,math
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,3,3,1,1,3,4,5,5,6,math
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,3,2,2,3,3,10,7,8,10,math
3,GP,F,15,U,GT3,T,4,2,health,services,...,2,2,1,1,5,2,15,14,15,math
4,GP,F,16,U,GT3,T,3,3,other,other,...,3,2,1,2,5,4,6,10,10,math


## Creating the risk label

In [6]:
df = combined_df.copy()

### Regression target
y_reg = df["G3"]

### Classification target: risk level from G3
def risk_label(g3):
    if g3 >= 15:
        return "Low"
    elif g3 >= 10:
        return "Medium"
    else:
        return "High"

df["risk_level"] = df["G3"].apply(risk_label)

print(df["risk_level"].value_counts())

risk_level
Medium    610
High      230
Low       204
Name: count, dtype: int64


## Train Test Split

In [7]:
from sklearn.model_selection import train_test_split

### Features
selected_features = [
"G2",
"G1",
"absences",
"studytime",
"age",
"famrel",
"freetime",
"health",
"goout",
"failures"
]

X = df[selected_features]

### Targets
y_reg = df["G3"]
y_cls = df["risk_level"]

### Train-Test split
X_train, X_test, y_reg_train, y_reg_test, y_cls_train, y_cls_test = train_test_split(X,
    y_reg,
    y_cls,
    test_size=0.2,
    random_state=42,
    stratify=y_cls)

print("Training set size:", X_train.shape)
print("Test set size:", X_test.shape)

Training set size: (835, 10)
Test set size: (209, 10)


In [8]:
print(y_cls_train.value_counts(normalize=True))
print(y_cls_test.value_counts(normalize=True))

risk_level
Medium    0.584431
High      0.220359
Low       0.195210
Name: proportion, dtype: float64
risk_level
Medium    0.583732
High      0.220096
Low       0.196172
Name: proportion, dtype: float64


## Data Preprocessing

In [9]:
### Identifying categorical and numerical columns
categorical_cols = X_train.select_dtypes(include=["object"]).columns
numerical_cols = X_train.select_dtypes(exclude=["object"]).columns

In [10]:
print("Categorical columns:", len(categorical_cols))
print("Numerical columns:", len(numerical_cols))

Categorical columns: 0
Numerical columns: 10


In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

### Encoding categorical columns
preprocessor = ColumnTransformer(transformers = [("cat", OneHotEncoder(handle_unknown = "ignore"), categorical_cols), ("num", StandardScaler(), numerical_cols)], remainder="drop")

In [12]:
print(preprocessor)

ColumnTransformer(transformers=[('cat', OneHotEncoder(handle_unknown='ignore'),
                                 Index([], dtype='object')),
                                ('num', StandardScaler(),
                                 Index(['G2', 'G1', 'absences', 'studytime', 'age', 'famrel', 'freetime',
       'health', 'goout', 'failures'],
      dtype='object'))])


In [13]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [14]:
print("Processed train shape:", X_train_processed.shape)
print("Processed test shape:", X_test_processed.shape)

Processed train shape: (835, 10)
Processed test shape: (209, 10)


## Linear Regression

In [15]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

### Pipeline
linreg_pipeline = Pipeline([("preprocessor", preprocessor), ("model", LinearRegression())])

### Train
linreg_pipeline.fit(X_train, y_reg_train)

### Predict
y_pred_lin = linreg_pipeline.predict(X_test)

### Evaluate
mae_lin = mean_absolute_error(y_reg_test, y_pred_lin)
rmse_lin = np.sqrt(mean_squared_error(y_reg_test, y_pred_lin))
r2_lin = r2_score(y_reg_test, y_pred_lin)

In [16]:
print("Linear Regression Results:")
print("MAE:", mae_lin)
print("RMSE:", rmse_lin)
print("R2:", r2_lin)

Linear Regression Results:
MAE: 0.9838664589921383
RMSE: 1.591298706459043
R2: 0.8279690321322399


## Random Forest Regression

In [17]:
from sklearn.ensemble import RandomForestRegressor

rf_reg_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators = 200,
        random_state = 42,
        n_jobs = -1
    ))
])

rf_reg_pipeline.fit(X_train, y_reg_train)

y_pred_rf = rf_reg_pipeline.predict(X_test)

mae_rf = mean_absolute_error(y_reg_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_reg_test, y_pred_rf))
r2_rf = r2_score(y_reg_test, y_pred_rf)

## Feature Selection

In [18]:
# Get the preprocessor from the pipeline
preprocessor = rf_reg_pipeline.named_steps["preprocessor"]

# Get transformed feature names
feature_names = preprocessor.get_feature_names_out()

# Get trained RF model
rf_model = rf_reg_pipeline.named_steps["model"]

importances = rf_model.feature_importances_

import pandas as pd

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values(by="importance", ascending=False)

importance_df.head(20)

,feature,importance
0,num__G2,0.818732
2,num__absences,0.060731
4,num__age,0.020342
1,num__G1,0.019421
6,num__freetime,0.016692
8,num__goout,0.015960
5,num__famrel,0.013725
7,num__health,0.013548
3,num__studytime,0.013035
9,num__failures,0.007815


In [19]:
import matplotlib.pyplot as plt
import os

# Plot top features
top_features = importance_df.head(10)

plt.figure(figsize=(8,6))
plt.barh(top_features["feature"], top_features["importance"])
plt.gca().invert_yaxis()

plt.title("Top Feature Importances (Random Forest)")
plt.xlabel("Importance")

# Ensure folder exists
os.makedirs("../static/images", exist_ok=True)

# Save figure
plt.savefig("../static/images/feature_importance.png")

plt.close()

print("Feature importance chart saved.")

Feature importance chart saved.


In [20]:
print("Random Forest Regressor Results:")
print("MAE:", mae_rf)
print("RMSE:", rmse_rf)
print("R2:", r2_rf)

Random Forest Regressor Results:
MAE: 0.8963193779904306
RMSE: 1.2939088055655623
R2: 0.8862607002615945


## Logistic Regression (Baseline Model)

In [21]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

logreg_cls_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=2000))
])

logreg_cls_pipeline.fit(X_train, y_cls_train)

y_pred_log = logreg_cls_pipeline.predict(X_test)

print("Logistic Regression Accuracy:", accuracy_score(y_cls_test, y_pred_log))
print("\nConfusion Matrix:\n", confusion_matrix(y_cls_test, y_pred_log))
print("\nClassification Report:\n", classification_report(y_cls_test, y_pred_log))

Logistic Regression Accuracy: 0.8708133971291866

Confusion Matrix:
 [[ 36   0  10]
 [  0  36   5]
 [  6   6 110]]

Classification Report:
               precision    recall  f1-score   support

        High       0.86      0.78      0.82        46
         Low       0.86      0.88      0.87        41
      Medium       0.88      0.90      0.89       122

    accuracy                           0.87       209
   macro avg       0.86      0.85      0.86       209
weighted avg       0.87      0.87      0.87       209



## Random Forest Classification

In [22]:
from sklearn.ensemble import RandomForestClassifier

rf_cls_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced"  # helps because Medium is the largest class
    ))
])

rf_cls_pipeline.fit(X_train, y_cls_train)

y_pred_rf_cls = rf_cls_pipeline.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_cls_test, y_pred_rf_cls))
print("\nConfusion Matrix:\n", confusion_matrix(y_cls_test, y_pred_rf_cls))
print("\nClassification Report:\n", classification_report(y_cls_test, y_pred_rf_cls))

Random Forest Accuracy: 0.8516746411483254

Confusion Matrix:
 [[ 36   0  10]
 [  0  35   6]
 [  8   7 107]]

Classification Report:
               precision    recall  f1-score   support

        High       0.82      0.78      0.80        46
         Low       0.83      0.85      0.84        41
      Medium       0.87      0.88      0.87       122

    accuracy                           0.85       209
   macro avg       0.84      0.84      0.84       209
weighted avg       0.85      0.85      0.85       209



In [23]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import os

# Compute confusion matrix
cm = confusion_matrix(y_cls_test, y_pred_log)

# Plot confusion matrix
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=rf_cls_pipeline.named_steps["model"].classes_
)

disp.plot(cmap="Blues")
plt.title("Classification Confusion Matrix")

# Ensure directory exists
os.makedirs("../static/images", exist_ok=True)

# Save figure
plt.savefig("../static/images/confusion_matrix.png")

plt.close()

## Trained models

In [24]:
import joblib

joblib.dump(rf_reg_pipeline, "rf_reg_pipeline.pkl")
joblib.dump(logreg_cls_pipeline, "logreg_cls_pipeline.pkl")

['logreg_cls_pipeline.pkl']

## Generalization Check for Regression

In [25]:
### Checking overfitting for Linear Regression 

### Training predictions
y_pred_train_lin = linreg_pipeline.predict(X_train)

### Training metrics
r2_train_lin = r2_score(y_reg_train, y_pred_train_lin)
r2_test_lin = r2_lin  # already computed earlier

print("Linear Regression Train R2:", r2_train_lin)
print("Linear Regression Test R2:", r2_test_lin)

### Checking overfitting for Random Forest Regression 

### Training predictions
y_pred_train_rf = rf_reg_pipeline.predict(X_train)

### Training metrics
r2_train_rf = r2_score(y_reg_train, y_pred_train_rf)
r2_test_rf = r2_rf  # already computed earlier

print("Random Forest Train R2:", r2_train_rf)
print("Random Forest Test R2:", r2_test_rf)

Linear Regression Train R2: 0.8373419858547096
Linear Regression Test R2: 0.8279690321322399
Random Forest Train R2: 0.9767773775051618
Random Forest Test R2: 0.8862607002615945


## Generalization Check for Classification

In [26]:
from sklearn.metrics import f1_score

print("RF Train Accuracy:", rf_cls_pipeline.score(X_train, y_cls_train))
print("RF Test Accuracy:", rf_cls_pipeline.score(X_test, y_cls_test))

print("RF Train F1 (macro):", f1_score(y_cls_train, rf_cls_pipeline.predict(X_train), average="macro"))
print("RF Test F1 (macro):", f1_score(y_cls_test, y_pred_rf_cls, average="macro"))

RF Train Accuracy: 1.0
RF Test Accuracy: 0.8516746411483254
RF Train F1 (macro): 1.0
RF Test F1 (macro): 0.8389476272436687


In [27]:
import json
import os
from sklearn.metrics import f1_score

# Compute metrics using values already available in the notebook
metrics = {
    "regression_r2": float(r2_rf),
    "classification_accuracy": float(rf_cls_pipeline.score(X_test, y_cls_test)),
    "classification_f1": float(f1_score(y_cls_test, y_pred_rf_cls, average="macro"))
}

# Ensure models directory exists
os.makedirs("../models", exist_ok=True)

# Save metrics
with open("../models/model_metrics.json", "w") as f:
    json.dump(metrics, f, indent=4)

print("Model metrics saved successfully.")

Model metrics saved successfully.
